In [1]:
from task3_1 import *
import numpy as np
import matplotlib.pyplot as plt

# Saving time evolution of magnetisation for different Ts

In [7]:
def task_1():
    L = 40
    n_sweeps = 10000
    Ts = np.array([2.1, 2.3, 2.5])
    spin_sums = np.zeros((len(Ts), n_sweeps))
  
    for i in range(len(Ts)):
        T = Ts[i]
        _, spin_sum, _ = run_metropolis(L, n_sweeps, T)
        spin_sums[i] = spin_sum

    np.savez('M_time_nofield.npz', Ts=Ts, spin_sums=spin_sums)

In [8]:
# run to save data from task 1
task_1()

In [9]:
def task_2_1():
    L = 40
    n_sweeps = 12000
    n_samples = 300
    ave = 10000
    T_range = np.linspace(1, 4, n_samples)
            
    M_evolution = np.zeros(n_samples)
    E_evolution = np.zeros(n_samples)

    spin_snapshots = np.zeros((3, L, L))
    for i in tqdm(range(n_samples)):
        T = T_range[i]
        _, spin_sums, total_Es = run_metropolis(L, n_sweeps, T)
        M_evolution[i] = np.abs(np.mean(spin_sums[-ave:]))
        E_evolution[i] = np.mean(total_Es[-ave:])

    np.savez('E_M_data_pure.npz', T_range=T_range, M_evolution=M_evolution, E_evolution=E_evolution)

# Saving E / M data for task 2.2, plus snapshots

In [10]:
def task_2_2():
    L = 40

    T_s = np.array([2.1, 2.3, 2.5])
    spin_snapshots = np.zeros((len(T_s), L, L))   

    n_sweeps = 12000

    for i in range(len(T_s)):
        T = T_s[i]
        spins, _, _ = run_metropolis(L, n_sweeps, T)
        spin_snapshots[i] = spins

    np.savez('spin_snapshots_pure.npz', T_s=T_s, spin_snapshots=spin_snapshots)

In [11]:
task_2_1()

100%|██████████| 300/300 [03:05<00:00,  1.62it/s]


In [ ]:
task_2_2()

# Saving the estimate of T_c for different L, as well as the plot for L = 40

In [ ]:
def task_3():
    L_s = np.array([10, 20, 30, 40])
    T_cs = []
    T_res = 500
    C_storage = np.zeros((len(L_s), T_res))

    for i in range(len(L_s)):
        L = L_s[i]
        T_range, C, T_c = find_C(L=L, cutoff=2000, n_sweeps=12000, T_res=T_res)
        T_cs.append(T_c)
        C_storage[i] = C
    
    T_cs = np.array(T_cs)
    np.savez('estimate_tc.npz', L_s=L_s, T_cs=T_cs, T_range=T_range, C_storage=C_storage)

In [6]:
task_3()

100%|██████████| 500/500 [08:26<00:00,  1.01s/it]


# Show effects of p on E, M and C

In [3]:
data = np.load('estimate_tc.npz')
L_s = data['L_s']
T_cs = data['T_cs']
T_range = data['T_range']
print(f'Tcs for Ls {L_s} are {T_range[T_cs]}')

Tcs for Ls [10 20 30 40] are [2.27655311 2.32064128 2.26653307 2.2745491 ]


In [4]:
def task_4():
    p_values = np.array([0.01, 0.10, 0.20])
    L = 40
    n_sweeps = 12000
    n_samples = 300
    ave = 10000

    T_range = np.linspace(1, 4, n_samples)
    M_evolutions = np.zeros((len(p_values), n_samples))
    E_evolutions = np.zeros((len(p_values), n_samples))
    C_evolutions = np.zeros((len(p_values), n_samples)) 

    for i in range(len(p_values)):
        p = p_values[i]
        M_evolution = np.zeros(n_samples)
        E_evolution = np.zeros(n_samples)
        C_evolution = np.zeros(n_samples)
        forbidden_mask = create_forbidden_mask(L, p)
        for j in tqdm(range(n_samples)):
            T = T_range[j]
            _, spin_sums, total_Es = run_metropolis_forbidden_sites(L, n_sweeps, T, p=p, forbidden_mask=forbidden_mask)
            last_spin_sums = spin_sums[-ave:]
            last_total_Es = total_Es[-ave:]

            M_evolution[j] = np.abs(np.mean(last_spin_sums))
            E_evolution[j] = np.mean(last_total_Es)
            C_evolution[j] = (np.var(last_total_Es)) / (T**2) 

        M_evolutions[i] = M_evolution
        E_evolutions[i] = E_evolution
        C_evolutions[i] = C_evolution

    np.savez('E_M_C_data_impure2.npz', 
             T_range=T_range, 
             M_evolutions=M_evolutions, 
             E_evolutions=E_evolutions, 
             C_evolutions=C_evolutions, 
             p_values=p_values)

In [5]:
task_4()

100%|██████████| 300/300 [04:48<00:00,  1.04it/s]


# Simulated annealing simulations

In [ ]:
def task_5():
    L = 40
    T_max = 5
    T_min = 0.1
    n_sweeps = 1000
    n_steps = 1000
    p_values = np.array([0.01, 0.10, 0.20])
    samples = 3

    E_vals = np.zeros((len(p_values), samples))
    M_vals = np.zeros((len(p_values), samples))
    spin_pictures = np.zeros((len(p_values), L, L))
    mask_pictures = np.zeros((len(p_values), L, L))
    
    
    for i in range(len(p_values)):
        p = p_values[i]
        E_min = np.inf
        lowest_spin = None
        lowest_mask = None
        mask = create_forbidden_mask(L, p)

        for j in range(samples):
            spins, mag, E, mask = run_simluated_annealing(L, n_sweeps, T_max, T_min=T_min, n_steps=n_steps, p=p, forbidden_mask=mask)
            
            if E < E_min:
                E_min = E
                lowest_spin = spins
                lowest_mask = mask
            
            E_vals[i, j] = E
            M_vals[i, j] = np.abs(mag)
        spin_pictures[i] = lowest_spin
        mask_pictures[i] = lowest_mask

    
    np.savez('SA_data_impure.npz',
                E_vals=E_vals, 
                M_vals=M_vals, 
                p_values=p_values, 
                spin_pictures=spin_pictures, 
                mask_pictures=mask_pictures)

In [5]:
task_5()

100%|██████████| 1000/1000 [01:33<00:00, 10.69it/s]
